# Cookie Cats A/B Test: Gate at Level 30 vs Level 40

**Question:** Does moving the first progress gate from level 30 (`gate_30`, control) to level 40 (`gate_40`, treatment) change player retention and engagement?

- **Primary metric:** `retention_7`
- **Secondary metrics:** `retention_1`, `sum_gamerounds`
- **Settings:** α = 0.05, two-sided tests, 95% confidence intervals, intention-to-treat (all randomized players)

## Part 1 – Setup
### 1.1 Configuration
Edit `DATA_PATH` if your file sits somewhere else in Google Drive.

In [ ]:
# ---- Edit this path if needed ----
DATA_PATH = "/content/drive/MyDrive/cookie_cats.csv"

# Experiment settings
CONTROL, TREATMENT = "gate_30", "gate_40"
ALPHA = 0.05          # significance level (two-sided)
CONF_LEVEL = 0.95     # confidence level for all intervals
SEED = 42             # seed for bootstrap and permutation steps
N_BOOT = 5_000        # bootstrap / permutation resamples

### 1.2 Imports and plot style

In [ ]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import (
    proportions_ztest,
    confint_proportions_2indep,
    proportion_effectsize,
)
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=FutureWarning)

# Reproducibility: one seeded generator for all resampling
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

# Plot style
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "figure.dpi": 100,
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
})
PALETTE = {CONTROL: "#1f77b4", TREATMENT: "#ff7f0e"}

# Table display
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print(f"pandas {pd.__version__} | numpy {np.__version__} | seaborn {sns.__version__}")
print(f"Seed = {SEED}, alpha = {ALPHA}, bootstrap resamples = {N_BOOT:,}")

### 1.3 Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 1.4 Load the data

In [ ]:
def load_data(path):
    """Load the CSV; print a clear message and stop if the path is wrong."""
    if not os.path.isfile(path):
        print(f"ERROR: file not found at:\n   {path}")
        print("Check DATA_PATH in cell 1.1 (spelling, folder, and that Drive is mounted).")
        # Help locate the file: list CSVs in the top level of My Drive
        candidates = sorted(glob.glob("/content/drive/MyDrive/*.csv"))
        if candidates:
            print("\nCSV files found in My Drive:")
            for c in candidates[:20]:
                print("   ", c)
        raise FileNotFoundError(path)
    return pd.read_csv(path)

df = load_data(DATA_PATH)

print(f"Loaded {DATA_PATH}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

# Quick check against the expected size
EXPECTED_SHAPE = (90_189, 5)
if df.shape == EXPECTED_SHAPE:
    print("Shape matches the expected 90,189 x 5.")
else:
    print(f"Warning: expected {EXPECTED_SHAPE}, got {df.shape}. Confirm this is the right file.")

df.head()

## Part 2 – Data Integrity
### 2.1 Column types and a quality log
Every check below writes one line to a data quality log, printed at the end of the part.

In [ ]:
# Data quality log: one row per check
dq_log = []

def log_check(check, passed, detail):
    """Record a data quality check and print it."""
    status = "PASS" if passed else "FLAG"
    dq_log.append({"check": check, "status": status, "detail": detail})
    print(f"[{status}] {check}: {detail}")

print("Column types as loaded:")
print(df.dtypes.to_string())

### 2.2 Convert retention columns to boolean
`pd.read_csv` usually reads `TRUE`/`FALSE` as booleans already. If not (for example strings or 0/1), this cell converts them and flags any value it cannot map.

In [ ]:
BOOL_MAP = {
    "true": True, "false": False,
    "t": True, "f": False,
    "yes": True, "no": False,
    "1": True, "0": False,
}

def to_bool(series):
    """Convert a column to bool; return (converted, number of unmapped values)."""
    if pd.api.types.is_bool_dtype(series):
        return series, 0
    mapped = series.astype(str).str.strip().str.lower().map(BOOL_MAP)
    n_bad = int(mapped.isna().sum() - series.isna().sum())
    return mapped, n_bad

for col in ["retention_1", "retention_7"]:
    original_dtype = df[col].dtype
    converted, n_bad = to_bool(df[col])
    if n_bad == 0 and converted.notna().all():
        df[col] = converted.astype(bool)
        note = "already boolean" if original_dtype == bool else f"converted from {original_dtype}"
        log_check(f"{col} is boolean", True, note)
    else:
        df[col] = converted  # keep NaN for unmapped values so they show up as missing
        log_check(f"{col} is boolean", False, f"{n_bad:,} values could not be mapped to True/False")

### 2.3 Unique IDs, missing values, duplicates

In [ ]:
# userid uniqueness
n_ids = df["userid"].nunique()
n_dup_ids = len(df) - n_ids
log_check("userid is unique", n_dup_ids == 0,
          f"{n_ids:,} unique IDs across {len(df):,} rows ({n_dup_ids:,} repeated)")

# Missing values per column
missing = df.isna().sum()
n_missing = int(missing.sum())
detail = "no missing values" if n_missing == 0 else \
    ", ".join(f"{c}={m:,}" for c, m in missing.items() if m > 0)
log_check("No missing values", n_missing == 0, detail)

# Fully duplicated rows
n_dup_rows = int(df.duplicated().sum())
log_check("No duplicated rows", n_dup_rows == 0, f"{n_dup_rows:,} fully duplicated rows")

# A repeated userid would mean a player was counted twice (possibly in both groups)
if n_dup_ids > 0:
    dup_ids = df.loc[df["userid"].duplicated(keep=False), "userid"]
    in_both = df[df["userid"].isin(dup_ids)].groupby("userid")["version"].nunique().gt(1).sum()
    print(f"   Note: {in_both:,} repeated IDs appear in both groups. Review before analysis.")

### 2.4 Value checks on `version` and `sum_gamerounds`

In [ ]:
# version must contain exactly the two expected arms
expected_versions = {CONTROL, TREATMENT}
found_versions = set(df["version"].dropna().unique())
unexpected = found_versions - expected_versions
log_check("version has only gate_30 / gate_40", not unexpected and found_versions == expected_versions,
          f"found {sorted(found_versions)}" + (f"; unexpected {sorted(unexpected)}" if unexpected else ""))

# sum_gamerounds must be whole, non-negative numbers
rounds = df["sum_gamerounds"]
is_int = pd.api.types.is_integer_dtype(rounds)
log_check("sum_gamerounds is integer", is_int, f"dtype = {rounds.dtype}")

n_negative = int((rounds < 0).sum())
log_check("sum_gamerounds is non-negative", n_negative == 0, f"{n_negative:,} negative values")

# Zero-round players are kept (they were randomized); just report them
n_zero = int((rounds == 0).sum())
log_check("Zero-round players kept", True,
          f"{n_zero:,} players ({n_zero / len(df):.2%}) played 0 rounds")

# Extreme values: compare the maximum to the next highest value
top_two = rounds.nlargest(2).tolist()
ratio = top_two[0] / top_two[1] if top_two[1] > 0 else np.inf
log_check("No extreme outlier in sum_gamerounds", ratio < 5,
          f"max = {top_two[0]:,}, next highest = {top_two[1]:,} ({ratio:.1f}x); handled in Part 5")

print("\nsum_gamerounds summary:")
print(rounds.describe(percentiles=[0.25, 0.5, 0.75, 0.99, 0.999]).round(1).to_string())

### 2.5 Data quality log

In [ ]:
dq_table = pd.DataFrame(dq_log)
n_flags = (dq_table["status"] == "FLAG").sum()

print(f"Data quality log: {len(dq_table)} checks, {n_flags} flagged\n")
display(dq_table)

if n_flags == 0:
    print("All checks passed. The data is ready for analysis.")
else:
    flagged = ", ".join(dq_table.loc[dq_table["status"] == "FLAG", "check"])
    print(f"Flagged: {flagged}.")
    print("No rows are removed here. The primary analysis keeps every randomized player; "
          "flagged items are handled explicitly in later parts.")

## Part 3 – Experiment Validity
### 3.1 Players per version

In [ ]:
n_by_version = df["version"].value_counts().reindex([CONTROL, TREATMENT])
n_total = int(n_by_version.sum())

counts_table = pd.DataFrame({
    "players": n_by_version,
    "share": (n_by_version / n_total).map("{:.3%}".format),
})
display(counts_table)

gap = int(n_by_version[TREATMENT] - n_by_version[CONTROL])
print(f"{TREATMENT} has {abs(gap):,} {'more' if gap > 0 else 'fewer'} players than {CONTROL} "
      f"({abs(gap) / n_total:.2%} of all players).")

### 3.2 Sample ratio mismatch (SRM) test
Players were meant to be split 50/50. A chi-square goodness-of-fit test checks whether the observed split is consistent with that.
If it is not, the randomization or the logging may be broken, and every comparison that follows carries that caveat.

Many experimentation teams use a stricter threshold (p < 0.001) for SRM alarms, so both thresholds are reported.

In [ ]:
SRM_STRICT_ALPHA = 0.001  # common industry threshold for SRM alarms

def srm_test(observed, expected_ratio=(0.5, 0.5)):
    """Chi-square goodness-of-fit test of group sizes against the planned split."""
    observed = np.asarray(observed, dtype=float)
    expected = observed.sum() * np.asarray(expected_ratio)
    chi2, p = stats.chisquare(observed, f_exp=expected)
    return {"chi2": chi2, "p_value": p, "observed": observed, "expected": expected}

srm_result = srm_test(n_by_version.values)

print("Sample ratio mismatch test (expected 50/50)")
for name, obs, exp in zip(n_by_version.index, srm_result["observed"], srm_result["expected"]):
    print(f"  {name}: observed {obs:,.0f} | expected {exp:,.1f} | diff {obs - exp:+,.1f}")
print(f"  chi-square = {srm_result['chi2']:.3f}, df = 1, p-value = {srm_result['p_value']:.4f}")

# Plain-English interpretation, driven by the numbers
p = srm_result["p_value"]
srm_flag = p < ALPHA
srm_result["flag"] = srm_flag
print("\nInterpretation:")
if p < SRM_STRICT_ALPHA:
    print(f"  p = {p:.4f} is below the strict SRM threshold of {SRM_STRICT_ALPHA}. This is a serious mismatch.")
    print("  The assignment or data collection is likely flawed. Treat all results below with strong caution.")
elif p < ALPHA:
    print(f"  p = {p:.4f} is below alpha = {ALPHA}, so the split differs from 50/50 more than chance alone")
    print(f"  would usually produce. It is above the strict SRM threshold of {SRM_STRICT_ALPHA}, so this is a")
    print("  warning rather than a hard stop. We continue, but record it as a caveat for the conclusion.")
else:
    print(f"  p = {p:.4f} is above alpha = {ALPHA}. The split is consistent with 50/50; no SRM detected.")

### 3.3 Diagnostic: zero-round players by version
Players with 0 rounds never reached either gate, so the gate position could not have affected them.
Their share should be about the same in both groups. A large difference would point to an assignment or logging problem
behind the SRM. We keep these players in every analysis because they were randomized.

In [ ]:
zero_by_version = (
    df.assign(zero_rounds=df["sum_gamerounds"].eq(0))
      .groupby("version")["zero_rounds"]
      .agg(zero_round_players="sum", players="count")
      .reindex([CONTROL, TREATMENT])
)
zero_by_version["share"] = zero_by_version["zero_round_players"] / zero_by_version["players"]
display(zero_by_version.style.format({"zero_round_players": "{:,}", "players": "{:,}", "share": "{:.2%}"}))

n_zero_total = int(zero_by_version["zero_round_players"].sum())
print(f"Total zero-round players: {n_zero_total:,} ({n_zero_total / n_total:.2%} of all players)")

# Are zero-round shares similar across groups?
zero_ct = np.column_stack([
    zero_by_version["zero_round_players"],
    zero_by_version["players"] - zero_by_version["zero_round_players"],
])
chi2_z, p_z, _, _ = stats.chi2_contingency(zero_ct)
diff_pp = (zero_by_version.loc[TREATMENT, "share"] - zero_by_version.loc[CONTROL, "share"]) * 100
print(f"Difference ({TREATMENT} - {CONTROL}): {diff_pp:+.2f} pp | chi-square p-value = {p_z:.4f}")
if p_z < ALPHA:
    print("Interpretation: zero-round shares differ between groups. Because the gate cannot affect these players,")
    print("this suggests the SRM may come from how players were assigned or logged.")
else:
    print("Interpretation: zero-round shares are similar in both groups. This check shows no sign that the")
    print("imbalance comes from a specific type of player, though it cannot rule out other causes.")

### 3.4 Crosstab of `retention_1` vs `retention_7`
Checks how the two retention metrics relate. Some players return on day 7 without returning on day 1, so the metrics are related but not nested.

In [ ]:
ct = pd.crosstab(df["retention_1"], df["retention_7"], margins=True, margins_name="Total")
ct.index.name, ct.columns.name = "retention_1", "retention_7"
print("Counts:")
display(ct)

print("Share of all players:")
display((pd.crosstab(df["retention_1"], df["retention_7"], normalize="all") * 100)
        .round(2).rename_axis(index="retention_1", columns="retention_7"))

# Conditional day-7 retention, by version
cond = (df.groupby(["version", "retention_1"])["retention_7"].mean()
          .unstack().reindex([CONTROL, TREATMENT]))
cond.columns = ["P(ret_7 | not ret_1)", "P(ret_7 | ret_1)"]
print("Day-7 retention conditional on day-1 retention, by version:")
display(cond.style.format("{:.2%}"))

# Interpretation from the numbers
n_r7 = int(df["retention_7"].sum())
n_r7_only = int((df["retention_7"] & ~df["retention_1"]).sum())
r7_given_r1 = df.loc[df["retention_1"], "retention_7"].mean()
r7_given_not_r1 = df.loc[~df["retention_1"], "retention_7"].mean()
print(f"Interpretation: {n_r7:,} players were retained on day 7. Of these, {n_r7_only:,} "
      f"({n_r7_only / n_r7:.1%}) did not return on day 1.")
print(f"Players who returned on day 1 were {r7_given_r1 / r7_given_not_r1:.1f}x as likely to return on day 7 "
      f"({r7_given_r1:.1%} vs {r7_given_not_r1:.1%}). The metrics are strongly linked,")
print("so their test results are not independent. The Holm correction in Part 10 stays valid under this dependence.")

### 3.5 Validity summary

In [ ]:
validity = pd.DataFrame([
    {"check": "Group sizes",
     "result": f"{CONTROL}: {n_by_version[CONTROL]:,} | {TREATMENT}: {n_by_version[TREATMENT]:,}"},
    {"check": "SRM chi-square (50/50)",
     "result": f"chi2 = {srm_result['chi2']:.2f}, p = {srm_result['p_value']:.4f} -> "
               + ("mismatch at alpha 0.05" if srm_result["flag"] else "no mismatch")},
    {"check": "Zero-round share by group",
     "result": f"diff = {diff_pp:+.2f} pp, p = {p_z:.4f}"},
    {"check": "Zero-round players (kept)",
     "result": f"{n_zero_total:,} ({n_zero_total / n_total:.2%})"},
])
display(validity)

## Part 4 – Exploratory Data Analysis
Everything here is descriptive. Formal tests come in Parts 6–9.

### 4.1 Descriptive table by version

In [ ]:
GROUPS = [CONTROL, TREATMENT]
LINESTYLE = {CONTROL: "-", TREATMENT: "--"}  # second cue besides color
GATES = (30, 40)

def summarize(g):
    """Key descriptive statistics for one group of players."""
    r = g["sum_gamerounds"]
    return pd.Series({
        "players": len(g),
        "retention_1": g["retention_1"].mean(),
        "retention_7": g["retention_7"].mean(),
        "zero_round_share": r.eq(0).mean(),
        "rounds_mean": r.mean(),
        "rounds_std": r.std(),
        "rounds_p25": r.quantile(0.25),
        "rounds_median": r.median(),
        "rounds_p75": r.quantile(0.75),
        "rounds_p90": r.quantile(0.90),
        "rounds_p99": r.quantile(0.99),
        "rounds_max": r.max(),
    })

desc = pd.DataFrame({v: summarize(df[df["version"] == v]) for v in GROUPS})
desc["all_players"] = summarize(df)

# Readable formatting per row
fmt = {"players": "{:,.0f}", "retention_1": "{:.2%}", "retention_7": "{:.2%}",
       "zero_round_share": "{:.2%}", "rounds_mean": "{:.2f}", "rounds_std": "{:.1f}"}
desc_display = desc.apply(lambda row: row.map(fmt.get(row.name, "{:,.0f}").format), axis=1)
display(desc_display)

# Interpretation from the numbers
d1 = (desc.loc["retention_1", TREATMENT] - desc.loc["retention_1", CONTROL]) * 100
d7 = (desc.loc["retention_7", TREATMENT] - desc.loc["retention_7", CONTROL]) * 100
max_group = desc.loc["rounds_max", GROUPS].idxmax()
print("Interpretation (descriptive only):")
print(f"  Day-1 retention: {TREATMENT} minus {CONTROL} = {d1:+.2f} pp.")
print(f"  Day-7 retention: {TREATMENT} minus {CONTROL} = {d7:+.2f} pp.")
print(f"  Rounds are heavily right-skewed: overall mean {desc.loc['rounds_mean', 'all_players']:.1f} "
      f"vs median {desc.loc['rounds_median', 'all_players']:.0f}.")
print(f"  The maximum ({desc.loc['rounds_max', max_group]:,.0f} rounds) is in {max_group}, which inflates "
      f"that group's mean and standard deviation.")
print("  Medians and rank-based tests are therefore more reliable than means for rounds.")

### 4.2 Distribution of rounds: log-scale histogram and ECDF
The x-axis uses a symmetric log scale so that zero-round players stay visible.

In [ ]:
def add_gate_lines(ax, label=True):
    """Mark the level-30 and level-40 gate positions."""
    for x, ls in zip(GATES, [":", "-."]):
        ax.axvline(x, color="grey", linestyle=ls, linewidth=1.2,
                   label=f"{x} rounds" if label else None)

rounds_max = df["sum_gamerounds"].max()
# Whole-number bin edges, spaced evenly on a log scale (no empty bins at low counts)
bins = np.unique(np.r_[0, np.floor(np.logspace(0, np.log10(rounds_max + 1), 50))])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram: share of each group per bin
ax = axes[0]
for v in GROUPS:
    r = df.loc[df["version"] == v, "sum_gamerounds"]
    ax.hist(r, bins=bins, weights=np.ones(len(r)) / len(r), histtype="step",
            linewidth=2, linestyle=LINESTYLE[v], color=PALETTE[v], label=v)
ax.set_xscale("symlog", linthresh=1)
add_gate_lines(ax)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_title("Distribution of game rounds by version")
ax.set_xlabel("Game rounds in first 14 days (symlog scale)")
ax.set_ylabel("Share of players in bin")
ax.legend(title="Version")

# ECDF: share of players with at most x rounds
ax = axes[1]
for v in GROUPS:
    r = np.sort(df.loc[df["version"] == v, "sum_gamerounds"].values)
    ax.step(r, np.arange(1, len(r) + 1) / len(r), where="post", linewidth=2,
            linestyle=LINESTYLE[v], color=PALETTE[v], label=v)
ax.set_xscale("symlog", linthresh=1)
add_gate_lines(ax)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_title("ECDF of game rounds by version")
ax.set_xlabel("Game rounds in first 14 days (symlog scale)")
ax.set_ylabel("Share of players with ≤ x rounds")
ax.legend(title="Version", loc="lower right")

plt.tight_layout()
plt.show()

# Interpretation: ECDF gap at key points
print("Share of players with at most x rounds:")
for x in [0, 10, 30, 40, 100]:
    shares = {v: (df.loc[df["version"] == v, "sum_gamerounds"] <= x).mean() for v in GROUPS}
    print(f"  ≤ {x:>3} rounds: {CONTROL} {shares[CONTROL]:.1%} | {TREATMENT} {shares[TREATMENT]:.1%} "
          f"| diff {100 * (shares[TREATMENT] - shares[CONTROL]):+.2f} pp")
ks_gap = stats.ks_2samp(df.loc[df["version"] == CONTROL, "sum_gamerounds"],
                        df.loc[df["version"] == TREATMENT, "sum_gamerounds"]).statistic
print(f"The largest vertical gap between the two ECDFs is {ks_gap:.1%} of players. "
      + ("The distributions nearly overlap, so any engagement effect is small relative to the spread."
         if ks_gap < 0.02 else "The distributions visibly differ; Part 7 tests whether this is significant."))

### 4.3 Top rounds values

In [ ]:
TOP_N = 10
top_rounds = (df.nlargest(TOP_N, "sum_gamerounds")[["userid", "version", "sum_gamerounds"]]
                .reset_index(drop=True))
top_rounds.index = top_rounds.index + 1
top_rounds["x_next_value"] = (top_rounds["sum_gamerounds"] /
                              top_rounds["sum_gamerounds"].shift(-1)).round(2)
print(f"Top {TOP_N} players by rounds played ('x_next_value' = ratio to the next row):")
display(top_rounds)

first, second = top_rounds["sum_gamerounds"].iloc[:2]
rest_max_ratio = top_rounds["x_next_value"].iloc[1:-1].max()
print(f"Interpretation: the top value ({first:,}) is {first / second:.1f}x the second ({second:,}). "
      f"Among the other rows, no value is more than {rest_max_ratio:.2f}x the next one.")
if first / second > 5 and rest_max_ratio < 2:
    print("Only the single top value looks like a data error or bot. Part 5 builds sensitivity datasets for it.")
else:
    print("Several values stand out; Part 5 checks how much they affect the results.")

### 4.4 Churn curve
**Left:** share of players who stopped at exactly *x* rounds. **Right:** share of players who played at least *x* rounds (a survival curve).
Vertical lines mark 30 and 40 rounds. Rounds are not levels (players can replay levels), so the gates will not line up exactly with these lines.

In [ ]:
MAX_X = 200
x = np.arange(0, MAX_X + 1)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
survival = {}
for v in GROUPS:
    r = df.loc[df["version"] == v, "sum_gamerounds"]
    exact = r.value_counts().reindex(x, fill_value=0) / len(r)
    survival[v] = pd.Series([(r >= k).mean() for k in x], index=x)
    axes[0].plot(x, exact.values, linestyle=LINESTYLE[v], color=PALETTE[v], linewidth=2, label=v)
    axes[1].plot(x, survival[v].values, linestyle=LINESTYLE[v], color=PALETTE[v], linewidth=2, label=v)

axes[0].set_title(f"Where players stop: share with exactly x rounds (0–{MAX_X})")
axes[0].set_ylabel("Share of players")
axes[1].set_title(f"Churn curve: share still playing at x rounds (0–{MAX_X})")
axes[1].set_ylabel("Share of players with ≥ x rounds")
for ax in axes:
    add_gate_lines(ax)
    ax.set_xlabel("Game rounds in first 14 days")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=0 if ax is axes[1] else 1))
    ax.legend(title="Version")
plt.tight_layout()
plt.show()

# Interpretation: survival at the gates
print("Share of players who reached at least x rounds:")
for k in [1, 30, 40, 100]:
    s30, s40 = survival[CONTROL][k], survival[TREATMENT][k]
    print(f"  ≥ {k:>3} rounds: {CONTROL} {s30:.1%} | {TREATMENT} {s40:.1%} | diff {100 * (s40 - s30):+.2f} pp")
drop_30_40 = {v: survival[v][30] - survival[v][40] for v in GROUPS}
print(f"Between 30 and 40 rounds, {CONTROL} loses {drop_30_40[CONTROL]:.1%} of all players "
      f"and {TREATMENT} loses {drop_30_40[TREATMENT]:.1%}.")
lost_before_30 = 1 - survival[CONTROL][30]
lost_before_10 = 1 - survival[CONTROL][10]
print(f"In {CONTROL}, {lost_before_30:.1%} of players stop before 30 rounds, and {lost_before_10:.1%} stop before 10.")
print(f"So {lost_before_10 / lost_before_30:.0%} of the churn before the first gate happens in the first 10 rounds.")

### 4.5 Retention rate by rounds bucket
Rounds are measured after treatment, so this view is descriptive only and must not be used to compare the groups causally.

In [ ]:
BUCKET_EDGES = [-1, 0, 9, 29, 39, 49, 99, 199, 499, np.inf]
BUCKET_LABELS = ["0", "1–9", "10–29", "30–39", "40–49", "50–99", "100–199", "200–499", "500+"]
df_b = df.assign(rounds_bucket=pd.cut(df["sum_gamerounds"], bins=BUCKET_EDGES, labels=BUCKET_LABELS))

bucket_tbl = (df_b.groupby(["rounds_bucket", "version"], observed=True)
                  .agg(players=("userid", "size"),
                       retention_1=("retention_1", "mean"),
                       retention_7=("retention_7", "mean"))
                  .unstack("version"))
display(bucket_tbl.style.format({c: ("{:,}" if c[0] == "players" else "{:.1%}") for c in bucket_tbl.columns}))

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
width = 0.4
pos = np.arange(len(BUCKET_LABELS))
for ax, metric, title in zip(axes, ["retention_1", "retention_7"], ["Day-1", "Day-7"]):
    for i, v in enumerate(GROUPS):
        ax.bar(pos + (i - 0.5) * width, bucket_tbl[(metric, v)].values, width=width,
               color=PALETTE[v], edgecolor="white", linewidth=2, label=v,
               hatch=None if v == CONTROL else "//")
    ax.set_xticks(pos, BUCKET_LABELS, rotation=30)
    ax.set_title(f"{title} retention by rounds bucket")
    ax.set_xlabel("Rounds played in first 14 days")
    ax.set_ylabel(f"{title} retention rate")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    ax.legend(title="Version")
plt.tight_layout()
plt.show()

# Interpretation: overall pattern and largest group gap
r7_gap = (bucket_tbl[("retention_7", TREATMENT)] - bucket_tbl[("retention_7", CONTROL)]) * 100
biggest = r7_gap.abs().idxmax()
lo, hi = bucket_tbl[("retention_7", CONTROL)].iloc[[0, -1]]
print(f"Interpretation: day-7 retention rises steeply with rounds played "
      f"(from {lo:.1%} at {BUCKET_LABELS[0]} rounds to {hi:.1%} at {BUCKET_LABELS[-1]} in {CONTROL}).")
print(f"The largest day-7 gap between versions is in the {biggest} bucket ({r7_gap[biggest]:+.1f} pp, "
      f"{TREATMENT} minus {CONTROL}).")
print("These gaps are not causal: the gate can change how many rounds a player plays, which changes who lands in each bucket.")

## Part 5 – Outlier Handling
The **primary analysis always uses the full dataset** (`df`, all randomized players). Two sensitivity datasets test whether the extreme value changes any conclusion:

| Dataset | What changes | Players |
|---|---|---|
| `df` (primary) | nothing | all |
| `df_no_outlier` | the single extreme player is removed | all minus 1 |
| `df_capped` | `sum_gamerounds` capped at the pooled 99.9th percentile | all |

The cap is computed on both groups together, so the same rule applies to each version.

### 5.1 Identify the extreme outlier
It is found by rule, not by hardcoded ID: the top value is an outlier if it is more than `OUTLIER_RATIO` times the next highest value.

In [ ]:
OUTLIER_RATIO = 5  # same rule as the data quality check in Part 2

top2 = df["sum_gamerounds"].nlargest(2)
ratio_top = top2.iloc[0] / top2.iloc[1]
outlier_idx = [top2.index[0]] if ratio_top > OUTLIER_RATIO else []

if outlier_idx:
    print(f"Extreme outlier found ({ratio_top:.1f}x the next highest value):")
    display(df.loc[outlier_idx])
    o = df.loc[outlier_idx[0]]
    print(f"This {o['version']} player logged {o['sum_gamerounds']:,} rounds in 14 days "
          f"(about {o['sum_gamerounds'] / 14:,.0f} per day), which is implausible for a person.")
    print(f"Retained on day 1: {o['retention_1']} | day 7: {o['retention_7']}. "
          "Removing them changes one player's retention, so the effect on retention tests is negligible.")
else:
    print(f"No value exceeds {OUTLIER_RATIO}x the next highest; df_no_outlier will equal df.")

### 5.2 Build the sensitivity datasets

In [ ]:
# 1) Without the extreme outlier
df_no_outlier = df.drop(index=outlier_idx).copy()

# 2) Rounds capped at the pooled 99.9th percentile (winsorized on the right)
CAP_Q = 0.999
cap_value = df["sum_gamerounds"].quantile(CAP_Q)
df_capped = df.copy()
df_capped["sum_gamerounds"] = df_capped["sum_gamerounds"].clip(upper=cap_value)
n_capped = int((df["sum_gamerounds"] > cap_value).sum())

print(f"df_no_outlier: {len(df_no_outlier):,} players ({len(df) - len(df_no_outlier)} removed)")
print(f"df_capped:     {len(df_capped):,} players; cap = {cap_value:,.1f} rounds "
      f"(p{CAP_Q * 100:g}); {n_capped:,} players capped")
print("Players capped by version:")
print(df.loc[df["sum_gamerounds"] > cap_value, "version"].value_counts()
        .reindex(GROUPS, fill_value=0).to_string())

# Registry used by the robustness checks in Part 9
DATASETS = {
    "Primary (all players)": df,
    "Without extreme outlier": df_no_outlier,
    f"Rounds capped at p{CAP_Q * 100:g}": df_capped,
}

# Safety check: the primary dataset must be untouched
assert len(df) == n_total and df["sum_gamerounds"].max() == rounds_max, "Primary dataset was modified!"
print("\nPrimary dataset unchanged: OK")

### 5.3 Compare the three datasets

In [ ]:
def dataset_summary(data, name):
    """Rounds and retention summary per version for one dataset."""
    out = (data.groupby("version")
               .agg(players=("userid", "size"),
                    rounds_mean=("sum_gamerounds", "mean"),
                    rounds_std=("sum_gamerounds", "std"),
                    rounds_median=("sum_gamerounds", "median"),
                    rounds_max=("sum_gamerounds", "max"),
                    retention_1=("retention_1", "mean"),
                    retention_7=("retention_7", "mean"))
               .reindex(GROUPS))
    out = out.reset_index()
    out.insert(0, "dataset", name)
    return out

comparison = pd.concat([dataset_summary(d, n) for n, d in DATASETS.items()], ignore_index=True)
display(comparison.style.format({
    "players": "{:,}", "rounds_mean": "{:.2f}", "rounds_std": "{:.1f}", "rounds_median": "{:.0f}",
    "rounds_max": "{:,.0f}", "retention_1": "{:.3%}", "retention_7": "{:.3%}",
}).hide(axis="index"))

# Difference in mean rounds (treatment - control) under each dataset
mean_diffs = (comparison.pivot(index="dataset", columns="version", values="rounds_mean")
                        .reindex(DATASETS.keys())
                        .rename_axis(columns=None))
mean_diffs["diff (gate_40 - gate_30)"] = mean_diffs[TREATMENT] - mean_diffs[CONTROL]
print("Mean rounds by dataset:")
display(mean_diffs.style.format("{:.2f}"))

# Interpretation from the numbers
prim, no_out = [mean_diffs.iloc[i]["diff (gate_40 - gate_30)"] for i in (0, 1)]
std_ctrl = comparison.loc[comparison["version"] == CONTROL, "rounds_std"].tolist()
print("Interpretation:")
print(f"  Removing one player changes the mean-rounds difference from {prim:+.2f} to {no_out:+.2f} rounds"
      + (" and flips its sign." if np.sign(prim) != np.sign(no_out) else "."))
print(f"  {CONTROL}'s standard deviation drops from {std_ctrl[0]:.1f} to {std_ctrl[1]:.1f}, "
      "so a single row dominates the spread of the mean.")
med_shift = comparison.groupby("version")["rounds_median"].agg(lambda m: m.max() - m.min()).max()
r7_shift = comparison.groupby("version")["retention_7"].agg(lambda r: r.max() - r.min()).max() * 100
print(f"  Across datasets, medians move by at most {med_shift:.0f} rounds and day-7 retention by at most "
      f"{r7_shift:.3f} pp.")
if med_shift <= 1 and r7_shift < 0.01:
    print("  So rank-based tests and retention tests should be robust to the outlier; mean-based results are not.")
else:
    print("  The outlier also affects medians or retention, so check every result on the sensitivity datasets.")
print("  Part 7 uses the primary data with a rank-based test; Part 9 reruns key tests on all three datasets.")

### 5.4 Visual check: rounds distribution before and after

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
plot_df = pd.concat([d[["version", "sum_gamerounds"]].assign(dataset=n) for n, d in DATASETS.items()])
sns.boxplot(data=plot_df, x="dataset", y="sum_gamerounds", hue="version", palette=PALETTE,
            hue_order=GROUPS, gap=0.1, showfliers=True, flierprops={"markersize": 3, "alpha": 0.4}, ax=ax)
ax.set_yscale("symlog", linthresh=1)
ax.set_ylim(0, rounds_max * 3)  # rounds are never negative; leave headroom for the outlier
ax.axhline(cap_value, color="grey", linestyle=":", linewidth=1.2, label=f"Cap = {cap_value:,.0f} rounds")
ax.set_title("Game rounds by version under each dataset")
ax.set_xlabel("Dataset")
ax.set_ylabel("Game rounds in first 14 days (symlog scale)")
ax.legend(title="Version", loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

## Part 6 – Retention Tests
For each retention metric we run:
- a **two-proportion z-test** (`statsmodels`, pooled variance, two-sided)
- a **chi-square test of independence** on the 2×2 table (no continuity correction, so χ² = z² as a cross-check)
- the **absolute difference** in percentage points (`gate_40 − gate_30`) with a 95% Newcomb (Wilson-based) CI
- the **relative lift** (`gate_40 / gate_30 − 1`) with a 95% log-ratio CI

Negative differences mean `gate_40` retains fewer players than `gate_30`.

### 6.1 Reusable proportion-test helper

In [ ]:
from statsmodels.stats.proportion import proportion_confint

def proportion_test(data, metric, alpha=ALPHA):
    """Compare a boolean metric between TREATMENT and CONTROL. Returns a dict of results."""
    g = data.groupby("version")[metric].agg(["sum", "count"]).reindex([CONTROL, TREATMENT])
    x_c, n_c = int(g.loc[CONTROL, "sum"]), int(g.loc[CONTROL, "count"])
    x_t, n_t = int(g.loc[TREATMENT, "sum"]), int(g.loc[TREATMENT, "count"])
    p_c, p_t = x_c / n_c, x_t / n_t

    # Two-proportion z-test (treatment first, so z < 0 means treatment is lower)
    z, p_z = proportions_ztest([x_t, x_c], [n_t, n_c], alternative="two-sided")

    # Chi-square test on the 2x2 table
    table = np.array([[x_c, n_c - x_c], [x_t, n_t - x_t]])
    chi2, p_chi2, _, _ = stats.chi2_contingency(table, correction=False)

    # Absolute difference (pp) and relative lift, each with a CI
    diff_lo, diff_hi = confint_proportions_2indep(x_t, n_t, x_c, n_c, compare="diff",
                                                  method="newcomb", alpha=alpha)
    ratio_lo, ratio_hi = confint_proportions_2indep(x_t, n_t, x_c, n_c, compare="ratio",
                                                    method="log", alpha=alpha)

    # Per-group Wilson CIs (reused for the headline chart in Part 11)
    ci_c = proportion_confint(x_c, n_c, alpha=alpha, method="wilson")
    ci_t = proportion_confint(x_t, n_t, alpha=alpha, method="wilson")

    return {
        "metric": metric, "x_c": x_c, "n_c": n_c, "x_t": x_t, "n_t": n_t,
        "rate_c": p_c, "rate_t": p_t, "rate_c_ci": ci_c, "rate_t_ci": ci_t,
        "diff_pp": (p_t - p_c) * 100, "diff_ci_pp": (diff_lo * 100, diff_hi * 100),
        "lift": p_t / p_c - 1, "lift_ci": (ratio_lo - 1, ratio_hi - 1),
        "z": z, "p_z": p_z, "chi2": chi2, "p_chi2": p_chi2,
    }

def interpret_proportion_test(r, alpha=ALPHA):
    """Print the results of proportion_test in plain English."""
    lo, hi = r["diff_ci_pp"]
    llo, lhi = r["lift_ci"]
    print(f"=== {r['metric']} ===")
    print(f"  {CONTROL}: {r['x_c']:,}/{r['n_c']:,} = {r['rate_c']:.2%}  "
          f"(95% CI {r['rate_c_ci'][0]:.2%} to {r['rate_c_ci'][1]:.2%})")
    print(f"  {TREATMENT}: {r['x_t']:,}/{r['n_t']:,} = {r['rate_t']:.2%}  "
          f"(95% CI {r['rate_t_ci'][0]:.2%} to {r['rate_t_ci'][1]:.2%})")
    print(f"  Difference ({TREATMENT} - {CONTROL}): {r['diff_pp']:+.2f} pp  (95% CI {lo:+.2f} to {hi:+.2f} pp)")
    print(f"  Relative lift: {r['lift']:+.2%}  (95% CI {llo:+.2%} to {lhi:+.2%})")
    print(f"  z-test:     z = {r['z']:.3f}, p = {r['p_z']:.4f}")
    match = "matches" if np.isclose(r["chi2"], r["z"] ** 2, rtol=1e-6) else "MISMATCH, check inputs"
    print(f"  chi-square: chi2 = {r['chi2']:.3f}, p = {r['p_chi2']:.4f}  (z^2 = {r['z'] ** 2:.3f}, {match})")

    direction = "lower" if r["diff_pp"] < 0 else "higher"
    if r["p_z"] < alpha:
        print(f"  Interpretation: {r['metric']} is significantly {direction} with {TREATMENT} "
              f"(p = {r['p_z']:.4f} < {alpha}).")
        print(f"  Moving the gate to level 40 changes this metric by {r['diff_pp']:+.2f} pp; the true effect "
              f"is plausibly between {lo:+.2f} and {hi:+.2f} pp.")
    else:
        print(f"  Interpretation: no significant difference at alpha = {alpha} (p = {r['p_z']:.4f}). "
              f"{TREATMENT} is {abs(r['diff_pp']):.2f} pp {direction},")
        print(f"  but the 95% CI ({lo:+.2f} to {hi:+.2f} pp) includes zero, so the data are consistent "
              "with no effect.")
    print()

### 6.2 Primary metric: `retention_7`

In [ ]:
ret7 = proportion_test(df, "retention_7")
interpret_proportion_test(ret7)

### 6.3 Secondary metric: `retention_1`
The p-value here is unadjusted; the Holm correction for secondary metrics is applied in Part 10.

In [ ]:
ret1 = proportion_test(df, "retention_1")
interpret_proportion_test(ret1)

### 6.4 Summary table and effect-size plot

In [ ]:
retention_tests = {"retention_7": ret7, "retention_1": ret1}

def fmt_ci(ci, pct=False, digits=2):
    """Format a (low, high) interval."""
    return f"[{ci[0]:+.{digits}%}, {ci[1]:+.{digits}%}]" if pct else f"[{ci[0]:+.{digits}f}, {ci[1]:+.{digits}f}]"

retention_summary = pd.DataFrame([{
    "metric": m + (" (primary)" if m == "retention_7" else " (secondary)"),
    f"{CONTROL} rate": f"{r['rate_c']:.2%}",
    f"{TREATMENT} rate": f"{r['rate_t']:.2%}",
    "diff (pp)": f"{r['diff_pp']:+.2f}",
    "95% CI (pp)": fmt_ci(r["diff_ci_pp"]),
    "relative lift": f"{r['lift']:+.2%}",
    "95% CI (lift)": fmt_ci(r["lift_ci"], pct=True),
    "z-test p": f"{r['p_z']:.4f}",
    "chi-square p": f"{r['p_chi2']:.4f}",
    "significant (unadjusted)": "yes" if r["p_z"] < ALPHA else "no",
} for m, r in retention_tests.items()])
display(retention_summary.style.hide(axis="index"))

# Effect-size plot: difference in pp with 95% CI
fig, ax = plt.subplots(figsize=(10, 3.5))
labels = list(retention_tests.keys())
for i, m in enumerate(labels):
    r = retention_tests[m]
    lo, hi = r["diff_ci_pp"]
    ax.errorbar(r["diff_pp"], i, xerr=[[r["diff_pp"] - lo], [hi - r["diff_pp"]]], fmt="o",
                color=PALETTE[TREATMENT], ecolor="dimgray", elinewidth=2, capsize=6, markersize=9,
                label="Estimate with 95% CI" if i == 0 else None)
    ax.annotate(f"{r['diff_pp']:+.2f} pp  (p = {r['p_z']:.4f})", (r["diff_pp"], i),
                textcoords="offset points", xytext=(0, 12), ha="center", fontsize=10)
ax.axvline(0, color="black", linestyle="--", linewidth=1, label="No difference")
ax.set_yticks(range(len(labels)), labels)
ax.set_ylim(-0.6, len(labels) - 0.4)
ax.set_title(f"Retention difference: {TREATMENT} minus {CONTROL}")
ax.set_xlabel("Difference in retention (percentage points)")
ax.set_ylabel("Metric")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

# Overall interpretation
sig = [m for m, r in retention_tests.items() if r["p_z"] < ALPHA]
print("Interpretation:")
if "retention_7" in sig:
    print(f"  The primary metric (retention_7) is significantly {'lower' if ret7['diff_pp'] < 0 else 'higher'} "
          f"with {TREATMENT}: {ret7['diff_pp']:+.2f} pp, or {ret7['lift']:+.1%} relative.")
else:
    print("  The primary metric (retention_7) shows no significant difference between versions.")
print(f"  retention_1: {ret1['diff_pp']:+.2f} pp (unadjusted p = {ret1['p_z']:.4f}); "
      "its Holm-adjusted verdict comes in Part 10.")
print("  Bootstrap (Part 8), permutation and Bayesian checks (Part 9) test how robust these results are.")

## Part 7 – Engagement Test (`sum_gamerounds`)
Rounds are heavily skewed and contain an extreme outlier, so a t-test on means is not appropriate. Instead:
- a **Mann-Whitney U test** (rank-based; asks whether a random `gate_40` player tends to play more or fewer rounds than a random `gate_30` player)
- **bootstrap 95% CIs** (percentile method, seed 42) for the difference in **means** and in **medians** (`gate_40 − gate_30`)

The primary dataset (all players, outlier included) is used here; Part 9 reruns this on the sensitivity datasets.

### 7.1 Mann-Whitney U test

In [ ]:
rounds_c = df.loc[df["version"] == CONTROL, "sum_gamerounds"].to_numpy()
rounds_t = df.loc[df["version"] == TREATMENT, "sum_gamerounds"].to_numpy()

def mann_whitney(a_treat, b_ctrl):
    """Two-sided Mann-Whitney U test with effect sizes."""
    u, p = stats.mannwhitneyu(a_treat, b_ctrl, alternative="two-sided")
    ps = u / (len(a_treat) * len(b_ctrl))  # P(treat > ctrl) + 0.5 * P(tie)
    return {"U": u, "p_value": p, "prob_superiority": ps, "rank_biserial": 2 * ps - 1}

mw = mann_whitney(rounds_t, rounds_c)
print("Mann-Whitney U test on sum_gamerounds")
print(f"  U = {mw['U']:,.0f}, p-value = {mw['p_value']:.4f}")
print(f"  Probability a random {TREATMENT} player plays more rounds than a random {CONTROL} player "
      f"(ties split): {mw['prob_superiority']:.4f}")
print(f"  Rank-biserial correlation: {mw['rank_biserial']:+.4f}  (0 = no effect, ±1 = complete separation)")

print("\nInterpretation:")
if mw["p_value"] < ALPHA:
    direction = "more" if mw["prob_superiority"] > 0.5 else "fewer"
    print(f"  The distributions differ significantly (p = {mw['p_value']:.4f}): {TREATMENT} players tend to play "
          f"{direction} rounds.")
else:
    print(f"  No significant difference in rounds played (p = {mw['p_value']:.4f} >= {ALPHA}).")
rb = abs(mw["rank_biserial"])
size_label = "negligible" if rb < 0.1 else "small" if rb < 0.3 else "medium" if rb < 0.5 else "large"
print(f"  The effect size is {size_label} (|r| = {rb:.3f}): "
      f"a random {TREATMENT} player out-plays a random {CONTROL} player {mw['prob_superiority']:.1%} of the time "
      "(50% = no difference).")

### 7.2 Bootstrap CIs for the difference in means and medians
Each resample draws players with replacement **within each group**, keeping the group sizes fixed.
The helper is generic, so Part 8 reuses it for retention. It creates its own generator from `SEED`, so results do not depend on cell run order.

In [ ]:
def bootstrap_diff(a_treat, b_ctrl, stat_fns, n_boot=N_BOOT, seed=SEED, chunk=250):
    """Bootstrap stat(treatment) - stat(control) for each function in stat_fns.

    stat_fns: dict name -> function applied along axis=1 of a 2-D resample array.
    Returns a dict name -> array of n_boot bootstrap differences.
    """
    gen = np.random.default_rng(seed)
    out = {name: [] for name in stat_fns}
    for start in range(0, n_boot, chunk):  # chunks keep memory low
        size = min(chunk, n_boot - start)
        rs_t = a_treat[gen.integers(0, len(a_treat), size=(size, len(a_treat)))]
        rs_c = b_ctrl[gen.integers(0, len(b_ctrl), size=(size, len(b_ctrl)))]
        for name, fn in stat_fns.items():
            out[name].append(fn(rs_t, axis=1) - fn(rs_c, axis=1))
    return {name: np.concatenate(v) for name, v in out.items()}

def percentile_ci(samples, conf=CONF_LEVEL):
    """Percentile bootstrap confidence interval."""
    tail = (1 - conf) / 2 * 100
    return tuple(np.percentile(samples, [tail, 100 - tail]))

boot_rounds = bootstrap_diff(rounds_t, rounds_c, {"mean": np.mean, "median": np.median})

engagement = {"mann_whitney": mw}
for stat_name, fn in [("mean", np.mean), ("median", np.median)]:
    observed = fn(rounds_t) - fn(rounds_c)
    lo, hi = percentile_ci(boot_rounds[stat_name])
    engagement[stat_name] = {"observed": observed, "ci": (lo, hi), "boot": boot_rounds[stat_name]}
    print(f"Difference in {stat_name} rounds ({TREATMENT} - {CONTROL}): {observed:+.2f}  "
          f"(95% bootstrap CI {lo:+.2f} to {hi:+.2f})")
    if lo > 0 or hi < 0:
        print(f"  -> The CI excludes 0: the {stat_name} differs between versions.")
    else:
        print(f"  -> The CI includes 0: no clear difference in the {stat_name}.")

print(f"\n({N_BOOT:,} bootstrap resamples, seed {SEED})")

### 7.3 Bootstrap distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, stat_name in zip(axes, ["mean", "median"]):
    res = engagement[stat_name]
    samples = res["boot"]
    if stat_name == "median":
        # Medians take only a few values, so show a bar per value
        vals, counts = np.unique(samples, return_counts=True)
        ax.bar(vals, counts, width=0.3, color=PALETTE[TREATMENT], alpha=0.75,
               edgecolor="white", label="Bootstrap differences")
    else:
        ax.hist(samples, bins=60, color=PALETTE[TREATMENT], alpha=0.75,
                edgecolor="white", label="Bootstrap differences")
    ax.axvline(res["observed"], color="black", linewidth=2, label=f"Observed: {res['observed']:+.2f}")
    for i, bound in enumerate(res["ci"]):
        ax.axvline(bound, color="dimgray", linestyle="--", linewidth=1.5,
                   label="95% CI" if i == 0 else None)
    ax.axvline(0, color="firebrick", linestyle=":", linewidth=1.5, label="No difference")
    ax.set_title(f"Bootstrap: difference in {stat_name} rounds ({TREATMENT} − {CONTROL})")
    ax.set_xlabel(f"Difference in {stat_name} rounds")
    ax.set_ylabel("Number of resamples")
    ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

# Interpretation: how the outlier shapes the mean distribution
out_group = df.loc[df["sum_gamerounds"].idxmax(), "version"]
n_out_group = (df["version"] == out_group).sum()
share_with_outlier = 1 - (1 - 1 / n_out_group) ** n_out_group
tail = "left" if out_group == CONTROL else "right"
print("Interpretation:")
print(f"  The extreme {out_group} player appears in about {share_with_outlier:.0%} of resamples. That stretches the")
print(f"  {tail} tail of the mean-difference distribution, so the mean CI is driven largely by one player.")
vals, counts = np.unique(engagement["median"]["boot"], return_counts=True)
print("  Medians take only a few values because rounds are whole numbers: "
      + ", ".join(f"{v:+.1f} ({c:,} resamples)" for v, c in zip(vals, counts)) + ".")

### 7.4 Engagement summary

In [ ]:
engagement_summary = pd.DataFrame([
    {"measure": "Mann-Whitney U (distribution shift)",
     "estimate": f"P(superiority) = {mw['prob_superiority']:.4f}",
     "95% CI": "n/a", "p-value": f"{mw['p_value']:.4f}"},
    {"measure": "Difference in mean rounds",
     "estimate": f"{engagement['mean']['observed']:+.2f}",
     "95% CI": fmt_ci(engagement["mean"]["ci"]), "p-value": "n/a (bootstrap)"},
    {"measure": "Difference in median rounds",
     "estimate": f"{engagement['median']['observed']:+.2f}",
     "95% CI": fmt_ci(engagement["median"]["ci"]), "p-value": "n/a (bootstrap)"},
])
display(engagement_summary.style.hide(axis="index"))

mw_sig = mw["p_value"] < ALPHA
mean_sig = not (engagement["mean"]["ci"][0] <= 0 <= engagement["mean"]["ci"][1])
median_sig = not (engagement["median"]["ci"][0] <= 0 <= engagement["median"]["ci"][1])
print("Interpretation:")
if not (mw_sig or mean_sig or median_sig):
    print("  None of the three measures shows a significant difference in rounds played.")
    print("  Moving the gate does not detectably change engagement in the first 14 days.")
else:
    which = [n for n, s in [("Mann-Whitney", mw_sig), ("mean", mean_sig), ("median", median_sig)] if s]
    print(f"  A significant difference appears in: {', '.join(which)}.")
    print("  Check Part 9 to see whether it survives removing or capping the outlier.")
# Flag borderline results: p just above alpha, or a CI that ends exactly at 0
borderline = []
if ALPHA <= mw["p_value"] < 2 * ALPHA:
    borderline.append(f"Mann-Whitney p = {mw['p_value']:.4f} is just above {ALPHA}")
for stat_name in ["mean", "median"]:
    if 0 in engagement[stat_name]["ci"]:
        borderline.append(f"the {stat_name} CI ends exactly at 0")
if borderline:
    print(f"  Borderline: {'; '.join(borderline)}.")
    print(f"  Treat this as a weak hint that {TREATMENT} players play slightly "
          f"{'fewer' if mw['prob_superiority'] < 0.5 else 'more'} rounds, not as evidence of an effect.")
print("  The Mann-Whitney p-value is the one carried into the Holm correction in Part 10.")

## Part 8 – Bootstrap Retention
A distribution-free check on Part 6. We resample players with replacement within each group (`N_BOOT` = 5,000, seed 42) and record the retention difference (`gate_40 − gate_30`) each time.

Both metrics are bootstrapped with the same seed and group sizes, so each resample uses **the same players** for `retention_1` and `retention_7`. This keeps the correlation between the two metrics, so we can also ask how often `gate_30` wins on both at once.

### 8.1 Run the bootstrap

In [ ]:
RET_METRICS = ["retention_1", "retention_7"]
boot_ret = {}

for m in RET_METRICS:
    a_t = df.loc[df["version"] == TREATMENT, m].to_numpy(dtype=np.int8)
    b_c = df.loc[df["version"] == CONTROL, m].to_numpy(dtype=np.int8)
    # Differences in percentage points
    boot_ret[m] = bootstrap_diff(a_t, b_c, {"mean": np.mean}, n_boot=N_BOOT, seed=SEED)["mean"] * 100

print(f"Bootstrap complete: {N_BOOT:,} resamples per metric (seed {SEED}).")

### 8.2 Bootstrap results

In [ ]:
boot_rows = []
for m in RET_METRICS:
    d = boot_ret[m]
    lo, hi = percentile_ci(d)
    boot_rows.append({
        "metric": m,
        "observed diff (pp)": retention_tests[m]["diff_pp"],
        "bootstrap mean (pp)": d.mean(),
        "bootstrap 95% CI (pp)": fmt_ci((lo, hi)),
        "Part 6 95% CI (pp)": fmt_ci(retention_tests[m]["diff_ci_pp"]),
        f"share {CONTROL} higher": (d < 0).mean(),
    })
boot_table = pd.DataFrame(boot_rows)
display(boot_table.style.format({
    "observed diff (pp)": "{:+.3f}", "bootstrap mean (pp)": "{:+.3f}", f"share {CONTROL} higher": "{:.2%}",
}).hide(axis="index"))

both = ((boot_ret["retention_1"] < 0) & (boot_ret["retention_7"] < 0)).mean()
print(f"Share of resamples in which {CONTROL} is higher on BOTH metrics: {both:.2%}")

print("\nInterpretation:")
for m in RET_METRICS:
    share = (boot_ret[m] < 0).mean()
    lo, hi = percentile_ci(boot_ret[m])
    ci_note = "excludes 0" if (hi < 0 or lo > 0) else "includes 0"
    strength = ("almost always" if share >= 0.99 else "in most resamples" if share >= 0.9 else
                "more often than not" if share > 0.5 else "less often than not")
    print(f"  {m}: {CONTROL} retains better in {share:.2%} of resamples ({strength}). "
          f"The bootstrap CI {ci_note}.")
max_gap = max(abs(percentile_ci(boot_ret[m])[i] - retention_tests[m]["diff_ci_pp"][i])
              for m in RET_METRICS for i in (0, 1))
print(f"  Bootstrap and Part 6 CI bounds differ by at most {max_gap:.3f} pp, "
      + ("so the analytic results are confirmed." if max_gap < 0.05 else "so compare both before concluding."))
print("  Note: the share of resamples is a measure of how stable the direction is, not a Bayesian probability;")
print("  the Bayesian posterior probability is computed in Part 9.")

### 8.3 Bootstrap distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
for ax, m in zip(axes, RET_METRICS):
    d = boot_ret[m]
    lo, hi = percentile_ci(d)
    share = (d < 0).mean()
    w = (d.max() - d.min()) / 60
    bins = np.arange(np.floor(d.min() / w) * w, d.max() + w, w)  # 0 falls on a bin edge
    # Split the histogram at zero: left = gate_30 higher, right = gate_40 higher
    ax.hist(d[d < 0], bins=bins, color=PALETTE[CONTROL], alpha=0.8, edgecolor="white",
            label=f"{CONTROL} higher ({share:.1%})")
    ax.hist(d[d >= 0], bins=bins, color=PALETTE[TREATMENT], alpha=0.8, edgecolor="white", hatch="//",
            label=f"{TREATMENT} higher ({1 - share:.1%})")
    ax.axvline(retention_tests[m]["diff_pp"], color="black", linewidth=2,
               label=f"Observed: {retention_tests[m]['diff_pp']:+.2f} pp")
    ax.axvline(lo, color="dimgray", linestyle="--", linewidth=1.5, label=f"95% CI [{lo:+.2f}, {hi:+.2f}]")
    ax.axvline(hi, color="dimgray", linestyle="--", linewidth=1.5)
    ax.axvline(0, color="firebrick", linestyle=":", linewidth=1.5, label="No difference")
    ax.set_title(f"Bootstrap: {m} difference ({TREATMENT} − {CONTROL})")
    ax.set_xlabel("Difference in retention (percentage points)")
    ax.set_ylabel("Number of resamples")
    ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()